# AI Engine - RAG Pipeline Flow

This notebook modularizes the RAG (Retrieval-Augmented Generation) pipeline into distinct logical steps as follows:

> **PDF Upload** → **Text Extraction** → **NLP Preprocessing** → **Chunking + Embeddings** → **Vector Database** → **User Query** → **Similarity Retrieval** → **Transformer Model (T5/BART)** → **Generated Answer** → **Chat Interface (MERN App)**

At the end, it binds these steps to a FastAPI server so your MERN app can communicate with it.

### Step 0: Imports, Setup, and Load Models
In this initial step, we import all necessary libraries and load our pre-trained machine learning models.

- **FastAPI & Uvicorn**: Used to spin up our API backend.
- **SentenceTransformer**: Embeds our text chunks into dense vectors.
- **Transformers (AutoModelForSeq2SeqLM, AutoTokenizer)**: Loads the T5 model for our generative QA tasks.
- **FAISS**: A library for efficient similarity search and clustering of dense vectors.
- **PyMuPDF (fitz)**: Extracts text from uploaded PDF files.

In [ ]:
from fastapi import FastAPI, UploadFile, File
from pydantic import BaseModel
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import faiss
import numpy as np
import fitz 
import os
import nest_asyncio
import uvicorn

nest_asyncio.apply()

print("Loading Models (Embeddings & T5)...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
print("[OK] Models Loaded!")

# Global State Variables for Vector Store
document_chunks = []
vector_index = None

### Step 1 & 2: PDF Upload & Text Extraction
When a user uploads a PDF document via the frontend, the raw bytes are processed here.

This function temporarily saves the uploaded PDF to disk, opens it using `PyMuPDF` (fitz), and iterates through each page to extract raw string text. This provides the fundamental knowledge base for the rest of the RAG pipeline.

In [ ]:
def extract_text_from_pdf(pdf_bytes: bytes) -> str:
    temp_path = "temp_uploaded.pdf"
    with open(temp_path, "wb") as f:
        f.write(pdf_bytes)
        
    doc = fitz.open(temp_path)
    text = "".join([page.get_text("text") + "\n" for page in doc])
    doc.close()
    os.remove(temp_path)
    return text

### Step 3: NLP Preprocessing & Chunking
Raw text from PDFs can be massive and unstructured. We cannot feed an entire book into our models at once due to token limits.

We clean the text to remove excessive whitespace and then use `RecursiveCharacterTextSplitter`. This breaks the text down into manageable chunks of 700 characters, with an overlap of 150 characters between chunks to preserve sentence context across boundaries.

In [ ]:
def preprocess_and_chunk(text: str) -> list:
    # Standard Preprocessing: Strip excess whitespace
    cleaned_text = "\n".join([line.strip() for line in text.split("\n") if line.strip()])
    
    # ENHANCEMENT: Increased chunk_size to 700 to prevent context loss.
    # 700 chars roughly matches the 256 token limit of our MiniLM embedding model.
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=150)
    chunks = text_splitter.split_text(cleaned_text)
    return chunks

### Step 4 & 5: Embeddings & Vector Database Setup
This step forms the 'Retrieval' part of RAG.

We run our text chunks through the SentenceTransformer model to convert them into mathematical vectors (embeddings). We then initialize a FAISS index, which acts as our high-speed Vector Database. By indexing these embeddings via Inner Product (which equates to Cosine Similarity when vectors are normalized), we can extremely quickly find the most relevant chunks later.

In [ ]:
def create_vector_db(chunks: list):
    # Generate embeddings and normalize
    chunk_embeddings = embedder.encode(chunks, normalize_embeddings=True)
    dimension = chunk_embeddings.shape[1]
    
    # Create FAISS Index with Inner Product for normalized Cosine Similarity
    index = faiss.IndexFlatIP(dimension) 
    index.add(np.array(chunk_embeddings))
    return index

### Step 6, 7 & 8: Intercepting the User Query, Similarity Retrieval, & AI Generation
When the user asks a question, this pipeline takes over:

1. **Embedding**: We turn the user's text question into a vector using the same SentenceTransformer.
2. **Similarity Search**: We search the FAISS index to pull out the Top 4 most relevant text chunks containing information related to the question.
3. **Generation**: We build a prompt feeding those 4 chunks to our `FLAN-T5` Language Model as context, asking it to synthesize a direct answer without hallucinating outside context.

In [ ]:
def answer_user_query(query: str, index, chunks: list) -> str:
    # 1. Similarity Retrieval 
    query_embedding = embedder.encode([query], normalize_embeddings=True)
    
    distances, indices = index.search(np.array(query_embedding), 4)
    
    retrieved_chunks = [chunks[indices[0][i]] for i in range(4)]
    context = "\n---\n".join(retrieved_chunks)
    
    # 2. Transformer Model Generation 
    # Using a clearer standard RAG prompt format
    prompt = f"Use the following context to answer the question accurately.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"
    
    
    inputs = tokenizer(prompt, return_tensors="pt", max_length=1024, truncation=True)
    outputs = model.generate(
        **inputs, 
        max_new_tokens=200, 
        do_sample=False, 
        repetition_penalty=1.2
    )
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return answer

### Step 9 & 10: Binding to FastAPI (MERN App Connections)
To make our standalone Python process accessible to our external Express.js backend, we wrap the entire pipeline into FastAPI Endpoint routes.

- `@app.post("/upload/")`: Executes Steps 1 through 5 whenever a new document is uploaded.
- `@app.post("/ask/")`: Executes Steps 6 through 8 whenever the user submits a chat message.
- Additional endpoints like `/summarize/`, `/literature-review/`, and `/key-points/` are provided for specialized extraction algorithms.

In [ ]:
app = FastAPI()

class QueryRequest(BaseModel):
    question: str

@app.post("/upload/")
async def upload_pdf_endpoint(file: UploadFile = File(...)):
    global document_chunks, vector_index
    
    # 1. & 2. Extractions
    pdf_bytes = await file.read()
    raw_text = extract_text_from_pdf(pdf_bytes)
    
    # 3. NLP Preprocessing & Chunking
    document_chunks = preprocess_and_chunk(raw_text)
    
    # 4. & 5. Vector Database
    vector_index = create_vector_db(document_chunks)
    
    return {"message": "Success"}

@app.post("/ask/")
def ask_question_endpoint(request: QueryRequest):
    # 6., 7., 8., 9.: Query Pipeline
    answer = answer_user_query(request.question, vector_index, document_chunks)
    
    # 10. Returning to MERN Interface
    return {"question": request.question, "answer": answer}

@app.get("/summarize/")
def summarize_document_endpoint():
    if not document_chunks:
        return {"summary": "No document uploaded."}
    
    context = document_chunks[0][:1000] 
    prompt = f"Summarize the following text concisely:\n\n{context}"
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    outputs = model.generate(**inputs, max_new_tokens=150)
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return {"summary": summary}

@app.get("/literature-review/")
def literature_review_endpoint():
    if not document_chunks:
        return {"review": "No document uploaded."}
    
    context = "\n---\n".join(document_chunks[:3]) 
    prompt = f"Provide a comprehensive literature review outlining the main themes and context of the following text:\n\n{context}"
    
    inputs = tokenizer(prompt, return_tensors="pt", max_length=1024, truncation=True)
    outputs = model.generate(**inputs, max_new_tokens=300, do_sample=False, repetition_penalty=1.2)
    review = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return {"review": review}

@app.get("/key-points/")
def key_points_endpoint():
    if not document_chunks:
        return {"key_points": "No document uploaded."}
    
    context = "\n---\n".join(document_chunks[:3]) 
    prompt = f"Extract the most important key points from the following text as a bulleted list:\n\n{context}"
    
    inputs = tokenizer(prompt, return_tensors="pt", max_length=1024, truncation=True)
    outputs = model.generate(**inputs, max_new_tokens=250, do_sample=False, repetition_penalty=1.2)
    key_points = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return {"key_points": key_points}


In [ ]:
# Start Server
print("Starting Chat Interface AI Engine on http://127.0.0.1:8000 ...")
from uvicorn import Config, Server
import asyncio

config = Config(app=app, host="127.0.0.1", port=8000)
server = Server(config=config)

# Using await directly bypasses the asyncio.run() conflict in Jupyter
await server.serve()